# Week 9 — Module 20: Bayesian Optimisation

Surrogate validation, acquisition comparisons, optimisation progress and function-specific refinement.

## Load observations

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from IPython.display import display

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Use the Week 8 observations for the Week 9 analysis.
search_roots = [Path.cwd(), *Path.cwd().parents]
relative_locations = [
    Path("8. Module 19/updated_data_week_8"),
    Path("updated_data_week_8"),
    Path("9. Module 20/Data"),
    Path("Data"),
]
data_folder = next(
    (base / location for base in search_roots for location in relative_locations
     if all((base / location / f"function_{i}.csv").is_file() for i in range(1, 9))),
    None,
)
if data_folder is None:
    raise FileNotFoundError("Could not find all eight Week 8 function CSVs. Place them in updated_data_week_8 or Data.")

data = {i: pd.read_csv(data_folder / f"function_{i}.csv") for i in range(1, 9)}
display(pd.DataFrame([{ "Function": f"F{i}", "Observations": len(data[i]), "Dimensions": data[i].shape[1]-1} for i in range(1,9)]))

,Function,Observations,Dimensions
0,F1,18,2
1,F2,18,2
2,F3,23,3
3,F4,38,4
4,F5,28,4
5,F6,28,5
6,F7,38,6
7,F8,48,8


## Week 8 observations

In [2]:
# Week 8 submitted points
week_8_points = {
    1: [0.998294, 0.000709],
    2: [0.697103, 0.994806],
    3: [0.463879, 0.497888, 0.452731],
    4: [0.386248, 0.381010, 0.343907, 0.423092],
    5: [1.000000, 1.000000, 0.970000, 1.000000],
    6: [0.426973, 0.384648, 0.468006, 0.752822, 0.084308],
    7: [0.157801, 0.305871, 0.468629, 0.230835, 0.339422, 0.619025],
    8: [0.026226, 0.080912, 0.128250, 0.056836,
        0.748250, 0.399531, 0.253527, 0.720750]
}

# Outputs returned for the Week 8 queries
week_8_outputs = {
    1: 0.0,
    2: 0.6571201027872348,
    3: -0.0008627356442611198,
    4: 0.3872619363215759,
    5: 8115.434723142181,
    6: -0.2743879717132347,
    7: 2.922059249051193,
    8: 9.957031536789
}

## Week 8 results

In [3]:
# Compare each new output with the previous observations.
week8_results = []
for i in range(1, 9):
    df = data[i]
    x_cols = list(df.columns[:-1])
    y_col = df.columns[-1]
    point = np.asarray(week_8_points[i], dtype=float)
    matching = np.all(np.isclose(df[x_cols].to_numpy(dtype=float), point, atol=5e-7, rtol=0), axis=1)
    historical = df.loc[~matching, y_col]
    if historical.empty:
        raise ValueError(f"F{i}: no historical observations available")
    previous_best = float(historical.max())
    output = float(week_8_outputs[i])
    week8_results.append({"Function": f"F{i}", "Previous best": previous_best,
                          "Week 8 output": output, "Difference": output - previous_best,
                          "Improved": output > previous_best})
display(pd.DataFrame(week8_results).round(6))

,Function,Previous best,Week 8 output,Difference,Improved
0,F1,0.000000,0.000000,-0.000000,False
1,F2,0.746962,0.657120,-0.089842,False
2,F3,-0.010190,-0.000863,0.009327,True
3,F4,0.186912,0.387262,0.200350,True
4,F5,8662.482500,8115.434723,-547.047777,False
5,F6,-0.379740,-0.274388,0.105352,True
6,F7,2.929370,2.922059,-0.007311,False
7,F8,9.930811,9.957032,0.026220,True


## Update observations

In [4]:
for i in range(1, 9):
    x_cols = list(data[i].columns[:-1])
    y_col = data[i].columns[-1]
    point = np.asarray(week_8_points[i], dtype=float)
    existing = data[i][x_cols].to_numpy(dtype=float)
    matching = np.flatnonzero(np.all(np.isclose(existing, point, atol=5e-7, rtol=0), axis=1))
    if len(matching):
        recorded = float(data[i].iloc[matching[0]][y_col])
        if not np.isclose(recorded, week_8_outputs[i], atol=1e-6, rtol=1e-8):
            raise ValueError(f"F{i}: recorded Week 8 output does not match")
    else:
        new_row = dict(zip(x_cols, point))
        new_row[y_col] = week_8_outputs[i]
        data[i] = pd.concat([data[i], pd.DataFrame([new_row])], ignore_index=True)
display(pd.DataFrame([{"Function": f"F{i}", "Observations": len(data[i]),
                      "Dimensions": data[i].shape[1]-1} for i in range(1,9)]))

,Function,Observations,Dimensions
0,F1,18,2
1,F2,18,2
2,F3,23,3
3,F4,38,4
4,F5,28,4
5,F6,28,5
6,F7,38,6
7,F8,48,8


## Running-best progress

In [5]:
for i in range(1, 9):

    y_col = data[i].columns[-1]
    y = data[i][y_col].to_numpy()

    # Running best after every observation
    running_best = np.maximum.accumulate(y)

    # Points at which a genuine new best occurred
    improvement_idx = np.where(
        np.r_[True, running_best[1:] > running_best[:-1]]
    )[0]


    # Show up to the five most recent occasions where a new best was found


rows = []
for i in range(1, 9):
    y = data[i].iloc[:, -1].to_numpy()
    running_best = np.maximum.accumulate(y)
    improvement_idx = np.where(np.r_[True, running_best[1:] > running_best[:-1]])[0]
    rows.append({"Function": f"F{i}", "Observations": len(y),
                 "Current best": running_best[-1], "New bests": len(improvement_idx),
                 "Recent bests": ', '.join(f'{running_best[j]:.6f}' for j in improvement_idx[-5:])})
display(pd.DataFrame(rows).round(6))


,Function,Observations,Current best,New bests,Recent bests
0,F1,18,0.000000,3,"0.000000, 0.000000, 0.000000"
1,F2,18,0.746962,3,"0.538996, 0.611205, 0.746962"
2,F3,23,-0.000863,8,"-0.020167, -0.016451, -0.015498, -0.010190, -0..."
3,F4,38,0.387262,11,"-4.025542, -0.300645, -0.200431, 0.186912, 0.3..."
4,F5,28,8662.482500,11,"4595.479318, 4857.449601, 6691.931674, 7121.15..."
5,F6,28,-0.274388,5,"-0.714265, -0.458511, -0.456874, -0.379740, -0..."
6,F7,38,2.929370,6,"1.364968, 1.950612, 2.139551, 2.536939, 2.929370"
7,F8,48,9.957032,9,"9.598482, 9.821986, 9.911249, 9.930811, 9.957032"


## Recent improvement sizes

In [6]:
for i in range(1, 9):

    y_col = data[i].columns[-1]
    y = data[i][y_col].to_numpy()

    running_best = np.maximum.accumulate(y)
    improvements = []

    for j in range(1, len(running_best)):
        if running_best[j] > running_best[j - 1]:

            previous_best = running_best[j - 1]
            new_best = running_best[j]
            gain = new_best - previous_best

            improvements.append(
                (j + 1, previous_best, new_best, gain)
            )


    if len(improvements) == 0:
        continue


rows = []
for i in range(1, 9):
    y = data[i].iloc[:, -1].to_numpy()
    running_best = np.maximum.accumulate(y)
    changes = np.where(running_best[1:] > running_best[:-1])[0] + 1
    for j in changes[-5:]:
        rows.append({"Function": f"F{i}", "Observation": j + 1,
                     "Previous best": running_best[j-1], "New best": running_best[j],
                     "Gain": running_best[j]-running_best[j-1]})
display(pd.DataFrame(rows).round(6))


,Function,Observation,Previous best,New best,Gain
0,F1,2,0.000000,0.000000,0.000000
1,F1,3,0.000000,0.000000,0.000000
2,F2,10,0.538996,0.611205,0.072209
3,F2,15,0.611205,0.746962,0.135757
4,F3,16,-0.034835,-0.020167,0.014668
5,F3,20,-0.020167,-0.016451,0.003716
6,F3,21,-0.016451,-0.015498,0.000953
7,F3,22,-0.015498,-0.010190,0.005309
8,F3,23,-0.010190,-0.000863,0.009327
9,F4,28,-6.702089,-4.025542,2.676547


## Gaussian Process models

In [7]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from sklearn.preprocessing import StandardScaler
import numpy as np

gp_models = {}
gp_scalers = {}


for i in range(1, 9):

    # Separate inputs and output
    X = data[i].iloc[:, :-1].values
    y = data[i].iloc[:, -1].values

    # Standardise outputs
    scaler = StandardScaler()
    y_scaled = scaler.fit_transform(y.reshape(-1, 1)).ravel()

    # Same RBF GP framework used previously
    kernel = RBF(
        length_scale=0.2,
        length_scale_bounds=(1e-5, 1e5)
    )

    gp = GaussianProcessRegressor(
        kernel=kernel,
        alpha=1e-6,
        normalize_y=False,
        n_restarts_optimizer=10,
        random_state=42
    )

    gp.fit(X, y_scaled)

    gp_models[i] = gp
    gp_scalers[i] = scaler


display(pd.DataFrame([{"Function": f"F{i}", "Observations": len(data[i]),
    "Kernel": str(gp_models[i].kernel_),
    "Length scale": float(gp_models[i].kernel_.length_scale),
    "Log marginal likelihood": gp_models[i].log_marginal_likelihood(gp_models[i].kernel_.theta)}
    for i in range(1,9)]).round(6))


,Function,Observations,Kernel,Length scale,Log marginal likelihood
0,F1,18,RBF(length_scale=1e-05),0.000010,-25.540894
1,F2,18,RBF(length_scale=1e-05),0.000010,-25.540894
2,F3,23,RBF(length_scale=0.201),0.201124,-25.852938
3,F4,38,RBF(length_scale=0.313),0.313209,-23.818726
4,F5,28,RBF(length_scale=0.303),0.303135,-23.970400
5,F6,28,RBF(length_scale=0.527),0.527186,-22.245983
6,F7,38,RBF(length_scale=0.323),0.323401,-34.151061
7,F8,48,RBF(length_scale=1.05),1.053652,-21.138922


## Neural-network LOOCV

In [8]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import LeaveOneOut
import numpy as np


nn_results = {}

for i in range(1, 9):

    X = data[i].iloc[:, :-1].values
    y = data[i].iloc[:, -1].values

    loo = LeaveOneOut()

    nn_predictions = []
    baseline_predictions = []
    actual_values = []

    for train_index, test_index in loo.split(X):

        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        # Scale inputs using training data only
        x_scaler = StandardScaler()
        X_train_scaled = x_scaler.fit_transform(X_train)
        X_test_scaled = x_scaler.transform(X_test)

        # Scale outputs using training data only
        y_scaler = StandardScaler()
        y_train_scaled = y_scaler.fit_transform(
            y_train.reshape(-1, 1)
        ).ravel()

        # Neural-network surrogate
        nn = MLPRegressor(
            hidden_layer_sizes=(50, 50),
            activation="relu",
            solver="lbfgs",
            max_iter=5000,
            random_state=42
        )

        nn.fit(X_train_scaled, y_train_scaled)

        # Predict and convert back to original output scale
        pred_scaled = nn.predict(X_test_scaled)

        pred = y_scaler.inverse_transform(
            pred_scaled.reshape(-1, 1)
        ).ravel()[0]

        # Mean baseline using training observations only
        baseline_pred = np.mean(y_train)

        nn_predictions.append(pred)
        baseline_predictions.append(baseline_pred)
        actual_values.append(y_test[0])

    # Calculate LOOCV errors
    nn_mae = mean_absolute_error(
        actual_values,
        nn_predictions
    )

    baseline_mae = mean_absolute_error(
        actual_values,
        baseline_predictions
    )

    beats_baseline = nn_mae < baseline_mae

    nn_results[i] = {
        "nn_mae": nn_mae,
        "baseline_mae": baseline_mae,
        "beats_baseline": beats_baseline
    }


display(pd.DataFrame([{"Function": f"F{i}", "NN LOOCV MAE": nn_results[i]["nn_mae"],
    "Baseline MAE": nn_results[i]["baseline_mae"],
    "Beats baseline": nn_results[i]["beats_baseline"]}
    for i in range(1,9)]).round(6))


,Function,NN LOOCV MAE,Baseline MAE,Beats baseline
0,F1,0.001837,0.000995,False
1,F2,0.257302,0.241635,False
2,F3,0.056881,0.053409,False
3,F4,1.385355,7.460801,True
4,F5,254.034588,2324.264043,True
5,F6,0.255143,0.530835,True
6,F7,0.444249,0.712291,True
7,F8,0.309128,0.981238,True


## GP acquisition candidates

In [9]:
from scipy.stats import norm
from scipy.spatial.distance import cdist

# Same candidate count and reproducible random generation as Week 8
n_candidates = 100000
rng = np.random.default_rng(42)

kappa_values = [0.5, 1.0, 1.5, 2.5]
xi_fractions = [0.00, 0.01, 0.05, 0.10]

gp_candidate_results = {}


for i in range(1, 9):

    X = data[i].iloc[:, :-1].values
    y = data[i].iloc[:, -1].values

    n_dimensions = X.shape[1]

    # Generate 100,000 candidates in [0,1]^d
    X_candidates = rng.random(
        (n_candidates, n_dimensions)
    )

    # GP predictions in scaled-output space
    mu_scaled, sigma_scaled = gp_models[i].predict(
        X_candidates,
        return_std=True
    )

    # Convert predictions back to original output scale
    scaler = gp_scalers[i]

    mu = scaler.inverse_transform(
        mu_scaled.reshape(-1, 1)
    ).ravel()

    sigma = sigma_scaled * scaler.scale_[0]

    # Candidate novelty
    nearest_distances = cdist(
        X_candidates,
        X
    ).min(axis=1)

    current_best = np.max(y)
    output_sd = np.std(y)

    function_results = {}


    # --------------------------------------------------
    # UCB
    # --------------------------------------------------


    for kappa in kappa_values:

        ucb = mu + kappa * sigma
        idx = np.argmax(ucb)

        function_results[f"UCB_{kappa}"] = {
            "candidate": X_candidates[idx],
            "predicted": mu[idx],
            "uncertainty": sigma[idx],
            "acquisition": ucb[idx],
            "nearest_distance": nearest_distances[idx]
        }


    # --------------------------------------------------
    # EXPECTED IMPROVEMENT
    # --------------------------------------------------


    for xi_fraction in xi_fractions:

        xi = xi_fraction * output_sd

        improvement = mu - current_best - xi

        with np.errstate(divide="ignore", invalid="ignore"):

            Z = np.divide(
                improvement,
                sigma,
                out=np.zeros_like(improvement),
                where=sigma > 0
            )

            ei = (
                improvement * norm.cdf(Z)
                + sigma * norm.pdf(Z)
            )

            ei[sigma <= 0] = 0

        idx = np.argmax(ei)

        label = f"EI_{int(xi_fraction * 100)}pct"

        function_results[label] = {
            "candidate": X_candidates[idx],
            "predicted": mu[idx],
            "uncertainty": sigma[idx],
            "acquisition": ei[idx],
            "nearest_distance": nearest_distances[idx],
            "xi": xi
        }


    gp_candidate_results[i] = function_results


rows = []
for i in range(1, 9):
    for method, r in gp_candidate_results[i].items():
        rows.append({"Function": f"F{i}", "Acquisition": method,
                     "Candidate": np.array2string(r["candidate"], precision=6, separator=", "),
                     "GP prediction": r["predicted"], "GP uncertainty": r["uncertainty"],
                     "Acquisition score": r["acquisition"], "Nearest distance": r["nearest_distance"]})
with pd.option_context("display.max_rows", 70, "display.max_columns", 10):
    display(pd.DataFrame(rows).round(6))


,Function,Acquisition,Candidate,GP prediction,GP uncertainty,Acquisition score,Nearest distance
0,F1,UCB_0.5,"[0.094177, 0.975622]",-0.000529,0.001544,0.000243,0.095343
1,F1,UCB_1.0,"[0.094177, 0.975622]",-0.000529,0.001544,0.001016,0.095343
2,F1,UCB_1.5,"[0.094177, 0.975622]",-0.000529,0.001544,0.001788,0.095343
3,F1,UCB_2.5,"[0.094177, 0.975622]",-0.000529,0.001544,0.003332,0.095343
4,F1,EI_0pct,"[0.094177, 0.975622]",-0.000529,0.001544,0.000387,0.095343
5,F1,EI_1pct,"[0.094177, 0.975622]",-0.000529,0.001544,0.000382,0.095343
6,F1,EI_5pct,"[0.094177, 0.975622]",-0.000529,0.001544,0.000360,0.095343
7,F1,EI_10pct,"[0.094177, 0.975622]",-0.000529,0.001544,0.000334,0.095343
8,F2,UCB_0.5,"[0.697103, 0.994806]",0.657023,0.006401,0.660224,0.000000
9,F2,UCB_1.0,"[0.697103, 0.994806]",0.657023,0.006401,0.663424,0.000000


## Neural-network candidates

In [10]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import cdist
import numpy as np


n_candidates = 100000
seeds = [42, 43, 44, 45, 46]

nn_candidate_results = {}
nn_models = {}

for i in range(1, 9):

    X = data[i].iloc[:, :-1].values
    y = data[i].iloc[:, -1].values

    n_dimensions = X.shape[1]

    # Scale complete dataset
    x_scaler = StandardScaler()
    X_scaled = x_scaler.fit_transform(X)

    y_scaler = StandardScaler()
    y_scaled = y_scaler.fit_transform(
        y.reshape(-1, 1)
    ).ravel()

    # Train NN ensemble
    models = []

    for seed in seeds:

        nn = MLPRegressor(
            hidden_layer_sizes=(50, 50),
            activation="relu",
            solver="lbfgs",
            max_iter=5000,
            random_state=seed
        )

        nn.fit(X_scaled, y_scaled)
        models.append(nn)

    nn_models[i] = {
        "models": models,
        "x_scaler": x_scaler,
        "y_scaler": y_scaler
    }

    # Same size candidate search as GP analysis
    rng = np.random.default_rng(42 + i)

    X_candidates = rng.random(
        (n_candidates, n_dimensions)
    )

    X_candidates_scaled = x_scaler.transform(
        X_candidates
    )

    # Predictions from each ensemble member
    predictions = []

    for model in models:

        pred_scaled = model.predict(
            X_candidates_scaled
        )

        pred = y_scaler.inverse_transform(
            pred_scaled.reshape(-1, 1)
        ).ravel()

        predictions.append(pred)

    predictions = np.array(predictions)

    mean_prediction = predictions.mean(axis=0)
    disagreement = predictions.std(axis=0)

    # Select candidate with highest mean prediction
    idx = np.argmax(mean_prediction)

    candidate = X_candidates[idx]
    predicted_output = mean_prediction[idx]
    candidate_disagreement = disagreement[idx]

    # Distance from existing observations
    nearest_distance = cdist(
        candidate.reshape(1, -1),
        X
    ).min()

    nn_candidate_results[i] = {
        "candidate": candidate,
        "predicted": predicted_output,
        "disagreement": candidate_disagreement,
        "nearest_distance": nearest_distance
    }


display(pd.DataFrame([{"Function": f"F{i}",
    "NN candidate": np.array2string(nn_candidate_results[i]["candidate"], precision=6, separator=", "),
    "NN prediction": nn_candidate_results[i]["predicted"],
    "Disagreement": nn_candidate_results[i]["disagreement"],
    "Nearest distance": nn_candidate_results[i]["nearest_distance"],
    "LOOCV validated": nn_results[i]["beats_baseline"]} for i in range(1,9)]).round(6))


,Function,NN candidate,NN prediction,Disagreement,Nearest distance,LOOCV validated
0,F1,"[5.179171e-01, 2.903274e-04]",0.001946,0.001331,0.182315,False
1,F2,"[0.998305, 0.758259]",0.998743,0.335442,0.122223,False
2,F3,"[0.465542, 0.513952, 0.462107]",-0.001015,0.000719,0.018674,False
3,F4,"[0.386457, 0.364301, 0.315578, 0.423391]",0.173431,0.101814,0.032892,True
4,F5,"[0.996673, 0.749297, 0.983441, 0.989318]",8191.723424,829.176761,0.247618,True
5,F6,"[0.4762 , 0.401879, 0.450627, 0.837966, 0.006...",-0.248980,0.042060,0.127622,True
6,F7,"[0.150161, 0.338209, 0.511025, 0.190914, 0.382...",2.893132,0.056058,0.110492,True
7,F8,"[0.012982, 0.180599, 0.189901, 0.256228, 0.760...",10.056825,0.027228,0.234486,True


## GP–NN candidate comparison

In [11]:
gp_nn_agreement = {}

for i in range(1, 9):

    nn_candidate = nn_candidate_results[i]["candidate"]

    comparisons = []

    for method, result in gp_candidate_results[i].items():

        gp_candidate = result["candidate"]

        distance = np.linalg.norm(
            nn_candidate - gp_candidate
        )

        comparisons.append(
            (method, distance, gp_candidate)
        )

    # Sort from closest to furthest
    comparisons.sort(key=lambda x: x[1])

    closest_method, closest_distance, closest_candidate = comparisons[0]

    gp_nn_agreement[i] = {
        "closest_method": closest_method,
        "distance": closest_distance,
        "gp_candidate": closest_candidate,
        "nn_candidate": nn_candidate
    }


display(pd.DataFrame([{"Function": f"F{i}",
    "Closest GP method": gp_nn_agreement[i]["closest_method"],
    "GP–NN distance": gp_nn_agreement[i]["distance"]} for i in range(1,9)]).round(6))


,Function,Closest GP method,GP–NN distance
0,F1,UCB_0.5,1.063404
1,F2,UCB_0.5,0.382985
2,F3,UCB_0.5,0.058180
3,F4,UCB_0.5,0.129899
4,F5,UCB_1.5,0.221396
5,F6,EI_0pct,0.193602
6,F7,UCB_0.5,0.125569
7,F8,UCB_0.5,0.243999


## Neighbourhood evidence

In [12]:
neighbourhood_results = {}

for i in range(1, 9):

    X = data[i].iloc[:, :-1].values
    y = data[i].iloc[:, -1].values

    candidates = {
        "NN": nn_candidate_results[i]["candidate"],
        "UCB_0.5": gp_candidate_results[i]["UCB_0.5"]["candidate"],
        "UCB_2.5": gp_candidate_results[i]["UCB_2.5"]["candidate"]
    }

    function_results = {}


    for method, candidate in candidates.items():

        distances = np.linalg.norm(
            X - candidate,
            axis=1
        )

        # Three nearest existing observations
        nearest_idx = np.argsort(distances)[:3]

        function_results[method] = {
            "candidate": candidate,
            "distances": distances[nearest_idx],
            "outputs": y[nearest_idx]
        }


    neighbourhood_results[i] = function_results


rows = []
for i in range(1, 9):
    for method, r in neighbourhood_results[i].items():
        rows.append({"Function": f"F{i}", "Method": method,
                     "Candidate": np.array2string(r["candidate"], precision=6, separator=", "),
                     "Nearest outputs": np.array2string(r["outputs"], precision=6, separator=", "),
                     "Nearest distances": np.array2string(r["distances"], precision=6, separator=", ")})
display(pd.DataFrame(rows))


,Function,Method,Candidate,Nearest outputs,Nearest distances
0,F1,NN,"[5.179171e-01, 2.903274e-04]","[-2.159249e-054, -2.089093e-091, 3.341771e-124]","[0.182315, 0.219703, 0.417006]"
1,F1,UCB_0.5,"[0.094177, 0.975622]","[0.000000e+00, 1.322677e-79, 1.033078e-46]","[0.095343, 0.309762, 0.489601]"
2,F1,UCB_2.5,"[0.094177, 0.975622]","[0.000000e+00, 1.322677e-79, 1.033078e-46]","[0.095343, 0.309762, 0.489601]"
3,F2,NN,"[0.998305, 0.758259]","[ 0.420586, 0.293993, -0.082041]","[0.122223, 0.160126, 0.283663]"
4,F2,UCB_0.5,"[0.697103, 0.994806]","[0.65712 , 0.611205, 0.36601 ]","[2.463010e-07, 6.846597e-02, 7.721404e-02]"
5,F2,UCB_2.5,"[0.403694, 0.391789]","[ 0.214965, -0.013858, -0.065624]","[0.113423, 0.189439, 0.264478]"
6,F3,NN,"[0.465542, 0.513952, 0.462107]","[-0.000863, -0.01019 , -0.016451]","[0.018674, 0.057502, 0.09955 ]"
7,F3,UCB_0.5,"[0.520345, 0.502063, 0.477604]","[-0.000863, -0.01019 , -0.016451]","[0.061843, 0.092797, 0.150761]"
8,F3,UCB_2.5,"[0.99839 , 0.992191, 0.910475]","[-0.056758, -0.113869, -0.105965]","[0.369217, 0.691881, 0.808789]"
9,F4,NN,"[0.386457, 0.364301, 0.315578, 0.423391]","[ 0.387262, 0.186912, -0.200431]","[0.032892, 0.035 , 0.07602 ]"


## Multi-radius local refinement

In [13]:
from scipy.spatial.distance import cdist
import numpy as np


local_functions = [3, 4, 5, 6, 7, 8]
radii = [0.03, 0.05, 0.10]
n_local = 50000

local_results = {}

for i in local_functions:

    X = data[i].iloc[:, :-1].values
    y = data[i].iloc[:, -1].values

    best_idx = np.argmax(y)
    best_point = X[best_idx]
    best_output = y[best_idx]

    function_results = {}


    for radius in radii:

        rng = np.random.default_rng(
            1000 + i + int(radius * 1000)
        )

        # Generate candidates around current best
        local_candidates = best_point + rng.uniform(
            -radius,
            radius,
            size=(n_local, X.shape[1])
        )

        local_candidates = np.clip(
            local_candidates,
            0,
            1
        )

        # Remove points too close to existing observations
        distances = cdist(local_candidates, X)
        nearest_distance = distances.min(axis=1)

        mask = nearest_distance > 1e-4

        local_candidates = local_candidates[mask]
        nearest_distance = nearest_distance[mask]

        # GP predictions
        mu_scaled, sigma_scaled = gp_models[i].predict(
            local_candidates,
            return_std=True
        )

        scaler = gp_scalers[i]

        gp_mu = scaler.inverse_transform(
            mu_scaled.reshape(-1, 1)
        ).ravel()

        gp_sigma = sigma_scaled * scaler.scale_[0]

        # NN ensemble predictions
        nn_info = nn_models[i]

        local_scaled = nn_info["x_scaler"].transform(
            local_candidates
        )

        nn_preds = []

        for model in nn_info["models"]:

            pred_scaled = model.predict(local_scaled)

            pred = nn_info["y_scaler"].inverse_transform(
                pred_scaled.reshape(-1, 1)
            ).ravel()

            nn_preds.append(pred)

        nn_preds = np.array(nn_preds)

        nn_mean = nn_preds.mean(axis=0)
        nn_disagreement = nn_preds.std(axis=0)

        # Normalise GP and NN predictions so that their
        # different output scales do not affect ranking
        gp_z = (
            (gp_mu - np.mean(gp_mu))
            / (np.std(gp_mu) + 1e-12)
        )

        nn_z = (
            (nn_mean - np.mean(nn_mean))
            / (np.std(nn_mean) + 1e-12)
        )

        # Equal cross-model support
        combined_score = gp_z + nn_z

        idx = np.argmax(combined_score)

        function_results[radius] = {
            "candidate": local_candidates[idx],
            "gp_predicted": gp_mu[idx],
            "gp_uncertainty": gp_sigma[idx],
            "nn_predicted": nn_mean[idx],
            "nn_disagreement": nn_disagreement[idx],
            "nearest_distance": nearest_distance[idx],
            "distance_from_best": np.linalg.norm(
                local_candidates[idx] - best_point
            )
        }

        result = function_results[radius]


    local_results[i] = function_results


rows = []
for i, settings in local_results.items():
    for radius, r in settings.items():
        rows.append({"Function": f"F{i}", "Radius": radius,
                     "Candidate": np.array2string(r["candidate"], precision=6, separator=", "),
                     "GP prediction": r["gp_predicted"], "GP uncertainty": r["gp_uncertainty"],
                     "NN prediction": r["nn_predicted"], "NN disagreement": r["nn_disagreement"],
                     "Nearest distance": r["nearest_distance"], "Distance from best": r["distance_from_best"]})
display(pd.DataFrame(rows).round(6))


,Function,Radius,Candidate,GP prediction,GP uncertainty,NN prediction,NN disagreement,Nearest distance,Distance from best
0,F3,0.03,"[0.473523, 0.500295, 0.456235]",-0.000166,0.001076,-0.000731,0.000640,0.010540,0.010540
1,F3,0.05,"[0.472143, 0.499213, 0.454876]",-0.000237,0.000874,-0.000683,0.000510,0.008640,0.008640
2,F3,0.10,"[0.473001, 0.501514, 0.457397]",-0.000225,0.001090,-0.000709,0.000636,0.010869,0.010869
3,F4,0.03,"[0.390756, 0.364086, 0.347307, 0.423281]",0.559706,0.041643,0.316357,0.047987,0.017842,0.017842
4,F4,0.05,"[0.388825, 0.366595, 0.350533, 0.420635]",0.563957,0.042578,0.311005,0.061416,0.016259,0.016259
5,F4,0.10,"[0.381791, 0.36297 , 0.347045, 0.425713]",0.531330,0.042915,0.326520,0.047721,0.019027,0.019027
6,F5,0.03,"[1. , 0.999897, 1. , 1. ]",8661.131159,2.884955,8659.521473,6.575730,0.000103,0.000103
7,F5,0.05,"[1. , 0.999876, 1. , 1. ]",8660.898026,2.939814,8659.550097,6.593814,0.000124,0.000124
8,F5,0.10,"[1. , 0.999875, 1. , 1. ]",8660.883409,2.943551,8659.551891,6.594977,0.000125,0.000125
9,F6,0.03,"[0.443441, 0.375612, 0.495655, 0.770339, 0.054...",-0.235263,0.009540,-0.253795,0.012928,0.048149,0.048149


## Function 5: novelty constraint

In [14]:
i = 5

X = data[i].iloc[:, :-1].values
y = data[i].iloc[:, -1].values

best_idx = np.argmax(y)
best_point = X[best_idx]
best_output = y[best_idx]

# Search broadly enough around the boundary optimum
search_radius = 0.15
n_local = 100000

novelty_thresholds = [0.03, 0.05, 0.10]

rng = np.random.default_rng(505)

local_candidates = best_point + rng.uniform(
    -search_radius,
    search_radius,
    size=(n_local, X.shape[1])
)

local_candidates = np.clip(
    local_candidates,
    0,
    1
)

# Distance from every existing observation
distances = cdist(local_candidates, X)
nearest_distances = distances.min(axis=1)

# GP predictions
mu_scaled, sigma_scaled = gp_models[i].predict(
    local_candidates,
    return_std=True
)

scaler = gp_scalers[i]

gp_mu = scaler.inverse_transform(
    mu_scaled.reshape(-1, 1)
).ravel()

gp_sigma = sigma_scaled * scaler.scale_[0]

# NN predictions
nn_info = nn_models[i]

local_scaled = nn_info["x_scaler"].transform(
    local_candidates
)

nn_preds = []

for model in nn_info["models"]:

    pred_scaled = model.predict(local_scaled)

    pred = nn_info["y_scaler"].inverse_transform(
        pred_scaled.reshape(-1, 1)
    ).ravel()

    nn_preds.append(pred)

nn_preds = np.array(nn_preds)

nn_mean = nn_preds.mean(axis=0)
nn_disagreement = nn_preds.std(axis=0)


f5_novelty_results = {}

for min_distance in novelty_thresholds:

    mask = nearest_distances >= min_distance

    valid_candidates = local_candidates[mask]
    valid_gp_mu = gp_mu[mask]
    valid_gp_sigma = gp_sigma[mask]
    valid_nn_mean = nn_mean[mask]
    valid_nn_disagreement = nn_disagreement[mask]
    valid_distances = nearest_distances[mask]

    # Standardise predictions within the valid candidate set
    gp_z = (
        (valid_gp_mu - valid_gp_mu.mean())
        / (valid_gp_mu.std() + 1e-12)
    )

    nn_z = (
        (valid_nn_mean - valid_nn_mean.mean())
        / (valid_nn_mean.std() + 1e-12)
    )

    combined_score = gp_z + nn_z

    idx = np.argmax(combined_score)

    candidate = valid_candidates[idx]

    f5_novelty_results[min_distance] = {
        "candidate": candidate,
        "gp_predicted": valid_gp_mu[idx],
        "gp_uncertainty": valid_gp_sigma[idx],
        "nn_predicted": valid_nn_mean[idx],
        "nn_disagreement": valid_nn_disagreement[idx],
        "nearest_distance": valid_distances[idx],
        "distance_from_best": np.linalg.norm(
            candidate - best_point
        )
    }

    result = f5_novelty_results[min_distance]


rows = []
for min_distance, r in f5_novelty_results.items():
    rows.append({"Min novelty": min_distance,
                 "Candidate": np.array2string(r["candidate"], precision=6, separator=", "),
                 "GP prediction": r["gp_predicted"], "GP uncertainty": r["gp_uncertainty"],
                 "NN prediction": r["nn_predicted"], "NN disagreement": r["nn_disagreement"],
                 "Nearest distance": r["nearest_distance"]})
display(pd.DataFrame(rows).round(6))


,Min novelty,Candidate,GP prediction,GP uncertainty,NN prediction,NN disagreement,Nearest distance
0,0.03,"[1. , 0.969998, 1. , 1. ]",8287.623017,247.689521,8701.926604,114.462640,0.030002
1,0.05,"[1. , 0.949994, 1. , 1. ]",8001.755491,410.333828,8731.544418,192.382555,0.050006
2,0.10,"[1. , 0.899981, 1. , 1. ]",7187.506528,802.299569,8783.689819,371.700555,0.100019


## Function 1: space-filling exploration

In [15]:
i = 1

X = data[i].iloc[:, :-1].values

# Large reproducible candidate set
rng = np.random.default_rng(101)
n_candidates = 200000

X_candidates = rng.random(
    (n_candidates, X.shape[1])
)

# Distance to nearest existing observation
distances = cdist(
    X_candidates,
    X
)

nearest_distances = distances.min(axis=1)

# Select the most isolated candidate
idx = np.argmax(nearest_distances)

f1_candidate = X_candidates[idx]
f1_nearest_distance = nearest_distances[idx]


# Show nearest observed points for context
nearest_idx = np.argsort(
    distances[idx]
)[:3]


display(pd.DataFrame([{"Function": "F1", "Candidate": np.array2string(f1_candidate, precision=6, separator=", "),
    "Nearest distance": f1_nearest_distance,
    "Three closest distances": np.array2string(np.sort(distances[idx])[:3], precision=6, separator=", ")}]).round(6))


,Function,Candidate,Nearest distance,Three closest distances
0,F1,"[0.682178, 0.001903]",0.306754,"[0.306754, 0.308314, 0.316118]"


## Function 2: NN-supported refinement

In [16]:
i = 2

X = data[i].iloc[:, :-1].values
y = data[i].iloc[:, -1].values

best_idx = np.argmax(y)
best_point = X[best_idx]
best_output = y[best_idx]

radii = [0.03, 0.05, 0.10]
n_local = 100000

f2_refinement_results = {}


for radius in radii:

    rng = np.random.default_rng(
        2000 + int(radius * 1000)
    )

    candidates = best_point + rng.uniform(
        -radius,
        radius,
        size=(n_local, X.shape[1])
    )

    candidates = np.clip(candidates, 0, 1)

    # Remove effectively duplicate observations
    nearest_distances = cdist(
        candidates,
        X
    ).min(axis=1)

    mask = nearest_distances > 1e-4

    candidates = candidates[mask]
    nearest_distances = nearest_distances[mask]

    # NN ensemble
    nn_info = nn_models[i]

    candidates_scaled = (
        nn_info["x_scaler"].transform(candidates)
    )

    predictions = []

    for model in nn_info["models"]:

        pred_scaled = model.predict(
            candidates_scaled
        )

        pred = nn_info["y_scaler"].inverse_transform(
            pred_scaled.reshape(-1, 1)
        ).ravel()

        predictions.append(pred)

    predictions = np.array(predictions)

    nn_mean = predictions.mean(axis=0)
    nn_disagreement = predictions.std(axis=0)

    # Select highest mean NN prediction
    idx = np.argmax(nn_mean)

    candidate = candidates[idx]

    f2_refinement_results[radius] = {
        "candidate": candidate,
        "nn_predicted": nn_mean[idx],
        "nn_disagreement": nn_disagreement[idx],
        "nearest_distance": nearest_distances[idx],
        "distance_from_best": np.linalg.norm(
            candidate - best_point
        )
    }

    result = f2_refinement_results[radius]


display(pd.DataFrame([{"Radius": radius,
    "Candidate": np.array2string(r["candidate"], precision=6, separator=", "),
    "NN prediction": r["nn_predicted"], "NN disagreement": r["nn_disagreement"],
    "Nearest distance": r["nearest_distance"], "Distance from best": r["distance_from_best"]}
    for radius, r in f2_refinement_results.items()]).round(6))


,Radius,Candidate,NN prediction,NN disagreement,Nearest distance,Distance from best
0,0.03,"[0.710015, 0.108158]",0.710872,0.113510,0.019873,0.030155
1,0.05,"[0.707188, 0.088281]",0.723824,0.159478,0.032572,0.050222
2,0.10,"[0.701892, 0.038659]",0.731854,0.249348,0.078369,0.100124


## Final validation and submission

In [17]:
import numpy as np
from scipy.spatial.distance import cdist


# Final selected candidates
final_candidates = {
    1: np.array([0.682178, 0.001903]),

    2: np.array([0.704305, 0.038527]),

    3: np.array([
        0.473523, 0.500295, 0.456235
    ]),

    4: np.array([
        0.388825, 0.366595, 0.350533, 0.420635
    ]),

    5: np.array([
        1.000000, 0.969998, 1.000000, 1.000000
    ]),

    6: np.array([
        0.440497, 0.372759, 0.525613,
        0.791828, 0.000000
    ]),

    7: np.array([
        0.199451, 0.330731, 0.460263,
        0.255214, 0.346569, 0.603374
    ]),

    8: np.array([
        0.031565, 0.175185, 0.160043,
        0.130670, 0.742356, 0.487463,
        0.194164, 0.622246
    ])
}

validation_results = {}

for i in range(1, 9):

    candidate = final_candidates[i]

    X = data[i].iloc[:, :-1].values

    expected_dimensions = X.shape[1]

    # Dimensionality
    correct_dimensions = (
        len(candidate) == expected_dimensions
    )

    # Bounds
    within_bounds = np.all(
        (candidate >= 0) & (candidate <= 1)
    )

    # Distances from every existing observation
    distances = cdist(
        candidate.reshape(1, -1),
        X
    ).ravel()

    nearest_distance = distances.min()
    nearest_index = np.argmin(distances)

    # Duplicate check
    duplicate = np.any(
        np.all(
            np.isclose(
                X,
                candidate,
                atol=1e-8,
                rtol=0
            ),
            axis=1
        )
    )

    passed = (
        correct_dimensions
        and within_bounds
        and not duplicate
    )

    validation_results[i] = {
        "dimensions": len(candidate),
        "expected_dimensions": expected_dimensions,
        "correct_dimensions": correct_dimensions,
        "within_bounds": within_bounds,
        "duplicate": duplicate,
        "nearest_distance": nearest_distance,
        "nearest_index": nearest_index,
        "passed": passed
    }


all_passed = all(
    result["passed"]
    for result in validation_results.values()
)


for i in range(1, 9):

    formatted = ", ".join(
        f"{x:.6f}"
        for x in final_candidates[i]
    )


display(pd.DataFrame([{"Function": f"F{i}",
    "Dimensions": validation_results[i]["dimensions"],
    "Within bounds": validation_results[i]["within_bounds"],
    "Duplicate": validation_results[i]["duplicate"],
    "Nearest distance": validation_results[i]["nearest_distance"],
    "Passed": validation_results[i]["passed"],
    "Submission": '-'.join(f'{v:.6f}' for v in final_candidates[i])}
    for i in range(1,9)]).round(6))

if not all(r["passed"] for r in validation_results.values()):
    raise ValueError("At least one final candidate failed validation")
print("Final submission points (six decimal places):")
for i in range(1,9):
    print(f"F{i}: " + '-'.join(f'{v:.6f}' for v in final_candidates[i]))


,Function,Dimensions,Within bounds,Duplicate,Nearest distance,Passed,Submission
0,F1,2,True,False,0.306755,True,0.682178-0.001903
1,F2,2,True,False,0.077676,True,0.704305-0.038527
2,F3,3,True,False,0.010539,True,0.473523-0.500295-0.456235
3,F4,4,True,False,0.016260,True,0.388825-0.366595-0.350533-0.420635
4,F5,4,True,False,0.030002,True,1.000000-0.969998-1.000000-1.000000
5,F6,5,True,False,0.110780,True,0.440497-0.372759-0.525613-0.791828-0.000000
6,F7,6,True,False,0.039502,True,0.199451-0.330731-0.460263-0.255214-0.346569-0...
7,F8,8,True,False,0.190714,True,0.031565-0.175185-0.160043-0.130670-0.742356-0...


Final submission points (six decimal places):
F1: 0.682178-0.001903
F2: 0.704305-0.038527
F3: 0.473523-0.500295-0.456235
F4: 0.388825-0.366595-0.350533-0.420635
F5: 1.000000-0.969998-1.000000-1.000000
F6: 0.440497-0.372759-0.525613-0.791828-0.000000
F7: 0.199451-0.330731-0.460263-0.255214-0.346569-0.603374
F8: 0.031565-0.175185-0.160043-0.130670-0.742356-0.487463-0.194164-0.622246
